# Momants topic classification

This notebook determines the main topic and subtopic of visitor messages. It does not perform sentiment, intent, or answer analysis. Only the six permitted Momants fields are loaded.

## 1. Import the topic module

In [1]:
from pathlib import Path
import importlib
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import momants_onderwerp
importlib.reload(momants_onderwerp)

<module 'momants_onderwerp' from '/home/runner/workspace/momants_onderwerp.py'>

## 2. Set the paths and event

In [2]:
CSV_PAD = PROJECT_DIR / "attached_assets" / "test_gesprekken_100.csv"
TOPICS_SEED_PATH = PROJECT_DIR / "momants_topics_seed_en.csv"
EVENT_ID = "decibel_2026"
OUTPUT_DIRECTORY = PROJECT_DIR / "results"

print(f"Input: {CSV_PAD}")
print(f"Topic seed: {TOPICS_SEED_PATH}")
print(f"Event: {EVENT_ID}")
print(f"Output directory: {OUTPUT_DIRECTORY}")

Input: /home/runner/workspace/attached_assets/test_gesprekken_100.csv
Topic seed: /home/runner/workspace/momants_topics_seed_en.csv
Event: decibel_2026
Output directory: /home/runner/workspace/results


## 3. Validate without loading the model

This checks the safe Momants loader and filters the active labels for the selected event.

In [3]:
data = momants_onderwerp.laad_momants_csv(CSV_PAD)
visitors = momants_onderwerp.selecteer_bezoekersberichten(data)
topics = momants_onderwerp.laad_onderwerpen(TOPICS_SEED_PATH, EVENT_ID)

print(f"Message rows: {len(data)}")
print(f"Usable visitor messages: {len(visitors)}")
print(f"Conversations: {visitors['conversation_id'].nunique()}")
print(f"Active topic labels: {len(topics)}")
display(topics[['main_topic', 'subtopic', 'description']])

Message rows: 265
Usable visitor messages: 143
Conversations: 96
Active topic labels: 36


,main_topic,subtopic,description
0,Tickets,Lost,"Lost ticket or ticket never received, e.g. due..."
1,Tickets,Rename,Questions about putting a ticket in someone el...
2,Tickets,Scanning,"Problems at the gates, entrance North/South sc..."
3,Tickets,Wristband,Wristband lost or broken
4,Tickets,AgeCheck,"ID checks, passport vs. driver's license, mini..."
5,Tickets,Upgrade,"Buying extra tickets, upgrading, or reselling ..."
6,Tickets,Other,Other questions about tickets and entry
7,Transport,Shuttle,"Shuttle bus times and stops, e.g. from Tilburg..."
8,Transport,Parking,"Getting there by car (exact address, Kiss & Ri..."
9,Transport,Transit,"Bike storage, scooters, public transport times"


## 4. Classify topics

The first zero-shot step selects one of six universal main topics or the internal `None` category. The second step compares the message only with event-specific descriptions belonging to the selected main topic, plus an automatically added `Other` candidate. `confidence` is the product of the winning main-topic and subtopic scores. Messages classified as `None` do not appear in the output.

In [4]:
topic_results = momants_onderwerp.process_csv(
    csv_path=CSV_PAD,
    seed_path=TOPICS_SEED_PATH,
    event_id=EVENT_ID,
    output_directory=OUTPUT_DIRECTORY,
    batch_size=16,
)

print(f"Complete: {len(topic_results)} conversation-topic combinations found.")
print(f"File: {topic_results.attrs['output_path']}")
topic_results.head(20)

Device set to use cpu


Complete: 136 conversation-topic combinations found.
File: /home/runner/workspace/results/topics_per_conversation_20260903_085235_291194.csv


,conversation_id,main_topic,subtopic,confidence,first_detected_at
0,0103cf17-7e89-4ad5-991c-b18d818ec166,Payments,Deposit,0.1759,2026-08-23T06:35:00+00:00
1,016b6ad0-4a3f-4643-a29d-ce2e97d9e92f,Payments,Deposit,0.1612,2026-08-22T09:38:00+00:00
2,016b6ad0-4a3f-4643-a29d-ce2e97d9e92f,Feedback,Other,0.3430,2026-08-22T09:38:50+00:00
3,01cf8b53-0970-41a1-a935-afc633d43918,Tickets,Rename,0.0872,2026-08-23T00:20:00+00:00
4,043a69ce-6eb8-4a18-b8cb-ff0f7d0e9c0f,Payments,Deposit,0.1848,2026-08-24T00:24:00+00:00
5,07043831-9ad6-45f6-b956-5ba5e63a9a92,Payments,Deposit,0.1759,2026-08-24T09:52:00+00:00
6,07043831-9ad6-45f6-b956-5ba5e63a9a92,Feedback,Other,0.2150,2026-08-24T09:53:00+00:00
7,08ea8523-94e5-4044-8327-28f444cf5f37,Tickets,Rename,0.1474,2026-08-22T13:50:00+00:00
8,0c6f5653-d431-4e90-8ec0-f6f59bfeed5b,Camping,Other,0.2444,2026-08-24T13:28:00+00:00
9,0c6f5653-d431-4e90-8ec0-f6f59bfeed5b,Feedback,Other,0.2150,2026-08-24T13:29:00+00:00
